In [1]:
import pandas as pd
import numpy as np
import bioframe as bf

### Changing prediction windows, and preventing from data leakage (as described in the paper)

In [2]:
ALPHA_INPUT_SIZE = 512 * 2048  # 1,048,576 bp (~1 Mb)
# FOLD_TO_PROCESS = "fold0"
# FOLD_TO_PROCESS = "fold1"
# FOLD_TO_PROCESS = "fold2"
FOLD_TO_PROCESS = "fold3"

In [3]:
# alphagenome_path = "/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/sequences_mouse.bed.gz"
alphagenome_path = "/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/sequences_human.bed.gz"

In [4]:
alpha_df = pd.read_csv(
    alphagenome_path,
    sep='\t',
    header=None,
    names=['chrom','start','end','type'],
    compression='gzip'
)

In [5]:
# Compute midpoint of target (196 kb) intervals
alpha_df['midpoint'] = (alpha_df['start'] + alpha_df['end']) // 2

# Compute 1 Mb input windows centered at midpoint
alpha_df['input_start'] = alpha_df['midpoint'] - ALPHA_INPUT_SIZE // 2
alpha_df['input_end'] = alpha_df['midpoint'] + ALPHA_INPUT_SIZE // 2
alpha_df['input_start'] = alpha_df['input_start'].clip(lower=0)  # no negative coords

In [ ]:
alpha_df

In [6]:
# valid_fold = "fold1"
# valid_fold = "fold2"
# valid_fold = "fold3"
valid_fold = "fold4"

In [7]:
# Define training and test intervals for FOLD_0 (moved by 1 for other models)
# According to AlphaGenome logic:
# - Test = fold0
# - Validation = fold1
# - Training = remaining 6 folds
train_df = alpha_df[~alpha_df['type'].isin([FOLD_TO_PROCESS, valid_fold])][['chrom','input_start','input_end','type']]
test_df = alpha_df[alpha_df['type'] == FOLD_TO_PROCESS][['chrom','input_start','input_end','type']]

In [8]:
test_df = test_df.rename(columns={'input_start':'start','input_end':'end'})
train_df = train_df.rename(columns={'input_start':'start','input_end':'end'})

In [ ]:
train_df

In [ ]:
test_df

In [9]:
df_overlap = bf.overlap(
        train_df, test_df, how="inner", suffixes=("_train", "_test"), cols1=["chrom", "start", "end"], cols2=["chrom", "start", "end"],
    )

In [ ]:
df_overlap

In [10]:
# Step 1: create a unique key for test windows
test_df['key'] = test_df[['chrom', 'start', 'end']].astype(str).agg('_'.join, axis=1)

# Step 2: create a key for overlapping windows
df_overlap['test_key'] = df_overlap[['chrom_test', 'start_test', 'end_test']].astype(str).agg('_'.join, axis=1)
overlap_keys = df_overlap['test_key'].unique()

# Step 3: keep only test windows NOT in overlap
test_safe_windows = test_df[~test_df['key'].isin(overlap_keys)].copy()
test_safe_windows.reset_index(drop=True, inplace=True)

print(f"Original test windows: {len(test_df)}")
print(f"Safe windows after exclusion: {len(test_safe_windows)}")

Original test windows: 6888
Safe windows after exclusion: 6323


In [11]:
# Add 'type' column
test_safe_windows['type'] = FOLD_TO_PROCESS

In [12]:
# Keep only desired columns
test_safe_windows_to_save = test_safe_windows[['chrom', 'start', 'end', 'type']]

In [13]:
test_safe_windows_to_save

,chrom,start,end,type
0,chr5,41200161,42248737,fold3
1,chr11,39963282,41011858,fold3
2,chr11,32144775,33193351,fold3
3,chr6,163839838,164888414,fold3
4,chr5,13310763,14359339,fold3
...,...,...,...,...
6318,chrX,40400309,41448885,fold3
6319,chr11,33865830,34914406,fold3
6320,chr6,125435725,126484301,fold3
6321,chr5,20498328,21546904,fold3


In [14]:
# Save to TSV
# test_safe_windows_to_save.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_{FOLD_TO_PROCESS}_safe.tsv", sep='\t', index=False)
test_safe_windows_to_save.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_{FOLD_TO_PROCESS}_safe.tsv", sep='\t', index=False)